# NyaayKhel — 03: Classifier Training & TFLite Export

**Purpose:** Train a GRU-based temporal classifier on the keypoint-sequence dataset built in
notebook 02, evaluate it with a confusion matrix, and export to TFLite for Android.

**Prerequisites:**
- `data/processed/X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy` exist (from notebook 02)
- `data/processed/label_map.json` exists

**What this notebook produces:**
- `model/nyaaykhel_classifier.pt` — PyTorch checkpoint
- `model/nyaaykhel_classifier.onnx` — ONNX export
- `model/nyaaykhel_classifier.tflite` — TFLite export (float32)
- `docs/confusion_matrix.png` — test set confusion matrix
- `docs/training_curves.png` — loss and accuracy over epochs
- `data/processed/model_config.json` — all hyperparameters for Android app to match

**Exit gate:** TFLite model exported, test accuracy logged, confusion matrix saved.


## Cell 1: Install & Imports


In [ ]:
!pip install -q onnx onnxruntime matplotlib seaborn

import os, json, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {DEVICE}')


## Cell 2: Paths & Load Dataset


In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/NyaayKhel'
else:
    BASE_DIR = '/content/NyaayKhel'

PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_DIR     = os.path.join(BASE_DIR, 'model')
DOCS_DIR      = os.path.join(BASE_DIR, 'docs')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

# Load arrays
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
X_test  = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

with open(os.path.join(PROCESSED_DIR, 'label_map.json')) as f:
    LABEL_MAP = json.load(f)
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = [INV_LABEL_MAP[i] for i in range(len(LABEL_MAP))]

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Classes: {CLASS_NAMES}')

WINDOW_SIZE  = X_train.shape[1]   # 30
FEATURE_DIM  = X_train.shape[2]   # 102
NUM_CLASSES  = len(LABEL_MAP)     # 4
print(f'Window: {WINDOW_SIZE} | Features: {FEATURE_DIM} | Classes: {NUM_CLASSES}')


## Cell 3: Model Architecture — GRU Classifier


In [ ]:
class KabaddiGRUClassifier(nn.Module):
    """
    Lightweight GRU-based temporal classifier for candidate kabaddi event detection.

    Architecture:
        Input: (batch, seq_len=30, feature_dim=102)
        GRU layer 1:   hidden_dim=128, batch_first=True
        GRU layer 2:   hidden_dim=128, batch_first=True (stacked)
        Dropout:       0.3 (regularisation for small dataset)
        FC layer:      128 -> num_classes=4
        Output:        (batch, num_classes) logits — softmax applied at inference

    Total parameters: ~165k — well within TFLite budget for low-end Android.

    Design rationale:
    - GRU (not LSTM) for fewer parameters and faster mobile inference
    - 2 stacked layers to capture both local motion (layer 1) and
      longer-range temporal patterns (layer 2)
    - Dropout at 0.3 to combat overfitting on the ~150-clip dataset
    - We use the last hidden state (not mean pool) — the final GRU state
      encodes the full sequence trajectory, which is what matters for event classification
    """
    def __init__(self, feature_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, feature_dim)
        out, hidden = self.gru(x)
        # Use last hidden state of top layer
        last_hidden = hidden[-1]          # (batch, hidden_dim)
        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)  # (batch, num_classes)
        return logits


# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
LEARNING_RATE = 1e-3
BATCH_SIZE   = 32
NUM_EPOCHS   = 60
RANDOM_SEED  = 42
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = KabaddiGRUClassifier(
    feature_dim=FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: KabaddiGRUClassifier')
print(f'  Feature dim: {FEATURE_DIM} | Hidden: {HIDDEN_DIM} | Layers: {NUM_LAYERS}')
print(f'  Total params:     {total_params:,}')
print(f'  Trainable params: {trainable_params:,}')
print(f'  Architecture:')
print(model)


## Cell 4: Training Setup — DataLoaders, Loss, Optimiser


In [ ]:
# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Class-weighted loss to handle imbalance
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
class_weights = torch.tensor(
    (class_counts.sum() / (NUM_CLASSES * class_counts)),
    dtype=torch.float32
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=8, factor=0.5, min_lr=1e-5
)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')
print(f'Class counts (train): {dict(zip(CLASS_NAMES, class_counts.astype(int)))}')
print(f'Class weights:        {dict(zip(CLASS_NAMES, class_weights.cpu().numpy().round(3)))}')
print(f'Loss: CrossEntropy (weighted) | Optimiser: Adam lr={LEARNING_RATE} wd=1e-4')
print(f'Scheduler: ReduceLROnPlateau patience=8 factor=0.5')


## Cell 5: Training Loop


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'lr': []}
best_test_acc = 0.0
best_model_path = os.path.join(MODEL_DIR, 'nyaaykhel_classifier_best.pt')

print(f'Training for {NUM_EPOCHS} epochs on {DEVICE}...')
print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>10}  {'Test Loss':>10}  {'Test Acc':>10}  {'LR':>8}')
print('-' * 68)

t_start = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss_sum += loss.item() * len(y_batch)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total   += len(y_batch)

    train_loss = train_loss_sum / train_total
    train_acc  = train_correct / train_total

    # ── Evaluate ─────────────────────────────────────────────────────────────
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)
    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['lr'].append(current_lr)

    # Save best model
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'test_acc': test_acc,
            'test_loss': test_loss,
        }, best_model_path)
        marker = ' *'
    else:
        marker = ''

    if epoch % 5 == 0 or epoch == 1 or marker:
        elapsed = time.time() - t_start
        print(f'{epoch:>6}  {train_loss:>10.4f}  {train_acc:>10.4f}  '
              f'{test_loss:>10.4f}  {test_acc:>10.4f}  {current_lr:>8.6f}{marker}')

print()
print(f'Training complete. Best test accuracy: {best_test_acc:.4f} ({best_test_acc*100:.1f}%)')
print(f'Best model saved: {best_model_path}')


## Cell 6: Plot Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train loss', linewidth=2)
axes[0].plot(epochs_range, history['test_loss'],  label='Test loss',  linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], label='Train acc', linewidth=2)
axes[1].plot(epochs_range, history['test_acc'],  label='Test acc',  linewidth=2)
axes[1].axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='70% target')
axes[1].axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='80% target')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle(f'NyaayKhel GRU Classifier Training\n'
             f'Best test acc: {best_test_acc*100:.1f}%',
             fontsize=13)
plt.tight_layout()

curves_path = os.path.join(DOCS_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {curves_path}')


## Cell 7: Confusion Matrix on Test Set


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Load best checkpoint
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best checkpoint (epoch {checkpoint["epoch"]}, test acc {checkpoint["test_acc"]:.4f})')

# Get all predictions
_, final_test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, DEVICE)
print(f'Final test accuracy: {final_test_acc*100:.1f}%')
print()

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')
axes[0].tick_params(axis='x', rotation=30)

# Row-normalised percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], vmin=0, vmax=100)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (% of true class)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'NyaayKhel GRU Classifier — Test Set\n'
             f'Overall accuracy: {final_test_acc*100:.1f}%', fontsize=13)
plt.tight_layout()

cm_path = os.path.join(DOCS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

# Per-class report
print()
print('Per-class classification report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Find worst confusions for model_eval.md
print('Most common confusions (top 3):')
confusions = []
for true_idx in range(NUM_CLASSES):
    for pred_idx in range(NUM_CLASSES):
        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:
            confusions.append((cm[true_idx, pred_idx],
                               CLASS_NAMES[true_idx], CLASS_NAMES[pred_idx]))
confusions.sort(reverse=True)
for count, true_name, pred_name in confusions[:3]:
    print(f'  True: {true_name:20s} -> Pred: {pred_name:20s}  ({count} instances)')


## Cell 8: Export to ONNX


In [ ]:
import onnx

model.eval()
onnx_path = os.path.join(MODEL_DIR, 'nyaaykhel_classifier.onnx')

# Dummy input matching training shape: (batch=1, seq_len=30, feature_dim=102)
dummy_input = torch.zeros(1, WINDOW_SIZE, FEATURE_DIM, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['keypoint_sequence'],
    output_names=['event_logits'],
    dynamic_axes={
        'keypoint_sequence': {0: 'batch_size'},
        'event_logits':      {0: 'batch_size'},
    },
)

# Verify ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

onnx_size_kb = os.path.getsize(onnx_path) / 1024
print(f'ONNX export: {onnx_path}')
print(f'ONNX size:   {onnx_size_kb:.1f} KB')
print('ONNX model check: PASSED')


## Cell 9: Convert ONNX → TFLite


In [ ]:
# Install onnx-tf (ONNX to TensorFlow converter)
!pip install -q onnx-tf tensorflow

import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')

TF_MODEL_DIR  = os.path.join(MODEL_DIR, 'tf_saved_model')
TFLITE_PATH   = os.path.join(MODEL_DIR, 'nyaaykhel_classifier.tflite')

# Step 1: ONNX -> TF SavedModel
print('Step 1: Converting ONNX -> TF SavedModel...')
onnx_model = onnx.load(onnx_path)
tf_rep = prepare(onnx_model)
tf_rep.export_graph(TF_MODEL_DIR)
print(f'  TF SavedModel saved: {TF_MODEL_DIR}')

# Step 2: TF SavedModel -> TFLite
print('Step 2: Converting TF SavedModel -> TFLite...')
converter = tf.lite.TFLiteConverter.from_saved_model(TF_MODEL_DIR)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,  # fallback for any non-standard ops
]
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # float32 with size optimisation
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

tflite_size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'  TFLite model saved: {TFLITE_PATH}')
print(f'  TFLite size: {tflite_size_kb:.1f} KB')


## Cell 10: Verify TFLite Inference & Speed


In [ ]:
import numpy as np
import tensorflow as tf
import time

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('TFLite Input tensor:')
for d in input_details:
    print(f'  name={d["name"]}  shape={d["shape"]}  dtype={d["dtype"]}')
print('TFLite Output tensor:')
for d in output_details:
    print(f'  name={d["name"]}  shape={d["shape"]}  dtype={d["dtype"]}')

# Functional check: run one window from test set
test_window = X_test[:1].astype(np.float32)  # (1, 30, 102)
interpreter.set_tensor(input_details[0]['index'], test_window)
interpreter.invoke()
tflite_logits = interpreter.get_tensor(output_details[0]['index'])  # (1, 4)
tflite_probs  = tf.nn.softmax(tflite_logits).numpy()
tflite_pred   = np.argmax(tflite_probs)
true_label    = int(y_test[0])

print()
print(f'Test window 0: true={CLASS_NAMES[true_label]}  '
      f'pred={CLASS_NAMES[tflite_pred]}  '
      f'conf={tflite_probs[0, tflite_pred]:.3f}')

# Speed benchmark: 100 windows
N_BENCH = min(100, len(X_test))
times = []
for i in range(N_BENCH):
    window = X_test[i:i+1].astype(np.float32)
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], window)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
    times.append(time.perf_counter() - t0)

print()
print(f'TFLite inference speed (Colab CPU, {N_BENCH} windows):')
print(f'  Avg: {np.mean(times)*1000:.1f}ms  Min: {np.min(times)*1000:.1f}ms  Max: {np.max(times)*1000:.1f}ms')
print(f'  Effective throughput: {1/np.mean(times):.0f} windows/sec')
print()
print('NOTE: Colab CPU ~ Android mid-range CPU. Low-end device may be 2-3x slower.')
target_fps = float(dataset_stats.get('target_fps', 10.0)) if 'dataset_stats' in globals() else 10.0
window_stride = int(dataset_stats.get('window_stride', WINDOW_SIZE // 3)) if 'dataset_stats' in globals() else 10
stride_seconds = window_stride / target_fps
print(f'With WINDOW_STRIDE={window_stride} frames at {target_fps:.1f}fps: new window every {stride_seconds:.2f}s.')
print(f'Target: classifier inference comfortably below {stride_seconds*1000:.0f}ms per window.')


## Cell 11: Save Model Config for Android App


In [ ]:
# This JSON is bundled with the Android app so it uses the exact same
# window size, feature dim, and class labels as the trained model.
# Path in app: android/app/src/main/assets/model_config.json

from sklearn.metrics import classification_report
import json

report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)

model_config = {
    # Input spec — must match Android EventClassifier.kt
    'window_size': WINDOW_SIZE,        # 30 frames
    'feature_dim': FEATURE_DIM,        # 102 = 2 persons x 17 kp x 3
    'max_persons': 2,
    'n_keypoints': 17,
    'person_conf_threshold':  0.5,
    'keypoint_conf_threshold': 0.3,

    # Output spec
    'num_classes': NUM_CLASSES,        # 4
    'class_names': CLASS_NAMES,        # index 0→raid_start, 1→touch, ...
    'label_map': LABEL_MAP,            # name→int
    'default_confidence_threshold': 0.65,  # events below this are not logged

    # Model files (relative to assets/ in Android)
    'pose_model_asset':       'yolov8n_pose.tflite',
    'classifier_model_asset': 'nyaaykhel_classifier.tflite',

    # Training provenance
    'architecture': 'GRU',
    'hidden_dim':   HIDDEN_DIM,
    'num_layers':   NUM_LAYERS,
    'total_params': total_params,
    'training_epochs': NUM_EPOCHS,
    'best_test_accuracy': float(final_test_acc),
    'training_data_source': 'YouTube public kabaddi footage (see docs/model_eval.md)',

    # Per-class metrics (for display in app / docs)
    'per_class_metrics': {
        cls: {
            'precision': round(report[cls]['precision'], 3),
            'recall':    round(report[cls]['recall'], 3),
            'f1':        round(report[cls]['f1-score'], 3),
        }
        for cls in CLASS_NAMES
    },
}

config_path = os.path.join(PROCESSED_DIR, 'model_config.json')
with open(config_path, 'w') as f:
    json.dump(model_config, f, indent=2)

# Also save alongside the TFLite model
with open(os.path.join(MODEL_DIR, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, indent=2)

print('model_config.json saved.')
print()
print('COPY THESE FILES TO Android app assets/ before Phase C:')
print(f'  {TFLITE_PATH}')
print(f'  {os.path.join(MODEL_DIR, "model_config.json")}')
print(f'  (also copy model/yolov8n_pose.tflite if not already there)')


## Cell 12: Save Final PyTorch Checkpoint


In [ ]:
final_pt_path = os.path.join(MODEL_DIR, 'nyaaykhel_classifier.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': model_config,
    'training_history': history,
    'class_names': CLASS_NAMES,
    'label_map': LABEL_MAP,
    'feature_dim': FEATURE_DIM,
    'window_size': WINDOW_SIZE,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'num_classes': NUM_CLASSES,
    'test_accuracy': float(final_test_acc),
}, final_pt_path)

print(f'Final PyTorch checkpoint: {final_pt_path}')
print(f'  Size: {os.path.getsize(final_pt_path)/1024:.1f} KB')


## Cell 13: Exit Gate & model_eval.md Fill-In


In [ ]:
from sklearn.metrics import classification_report
import json

checks = [
    ('PyTorch checkpoint saved',     os.path.exists(os.path.join(MODEL_DIR, 'nyaaykhel_classifier.pt'))),
    ('ONNX export saved',            os.path.exists(onnx_path)),
    ('TFLite model saved',           os.path.exists(TFLITE_PATH)),
    ('Confusion matrix PNG saved',   os.path.exists(os.path.join(DOCS_DIR, 'confusion_matrix.png'))),
    ('Training curves PNG saved',    os.path.exists(os.path.join(DOCS_DIR, 'training_curves.png'))),
    ('model_config.json saved',      os.path.exists(os.path.join(MODEL_DIR, 'model_config.json'))),
    ('Test accuracy >= 0.60',        final_test_acc >= 0.60),
    ('TFLite inference < 500ms avg', np.mean(times)*1000 < 500),
]

print('=' * 55)
print('PHASE B TRAINING EXIT GATE')
print('=' * 55)
all_pass = True
for label, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'  [{status}]  {label}')

print()
print(f'Final test accuracy: {final_test_acc*100:.1f}%')
print(f'TFLite size:         {os.path.getsize(TFLITE_PATH)/1024:.1f} KB')
print(f'TFLite avg latency:  {np.mean(times)*1000:.1f}ms (Colab CPU)')

if all_pass:
    print()
    print('ALL CHECKS PASSED. Proceed to Phase C (Android app).')
    print()
    print('FILL IN docs/model_eval.md with these values before Phase D:')
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)
    print(f'  Overall accuracy: {final_test_acc*100:.1f}%')
    print(f'  Total params:     {total_params:,}')
    print(f'  TFLite size:      {os.path.getsize(TFLITE_PATH)/1024:.1f} KB')
    print()
    print('  Per-class metrics:')
    for cls in CLASS_NAMES:
        m = report[cls]
        print(f'    {cls:20s}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1-score"]:.3f}')
    print()
    print('  Top confusions:')
    for count, true_name, pred_name in confusions[:3]:
        print(f'    {true_name} -> {pred_name}: {count} instances')
else:
    print()
    print('SOME CHECKS FAILED.')
    if final_test_acc < 0.60:
        print('  -> Accuracy below 60%. Try: more labeled data, more epochs, lower LR.')
    if np.mean(times)*1000 >= 500:
        print('  -> TFLite too slow. Try: smaller hidden_dim (64), fewer layers (1).')
